# Demo: Part 1 - Multimodal RAG
DATA 266 Lab 2, Pair_Programming_Team_32. Runs the graded code (tag `p1-v1.0`) live on questions the graders choose.

**Before the demo (about 5 min):** Colab, CPU runtime is enough. Secrets: `GITHUB_TOKEN`, `ANTHROPIC_API_KEY`
(`GEMINI_API_KEY` optional). Put the 3 PDFs in `MyDrive/genai_lab2/p1_data/` (or upload them when asked).
Run all cells down to **Ask questions**.

In [ ]:
# Cell 1 - run information: clone the graded tag, then print commit, versions, GPU and seed
import os, sys, subprocess, platform, random, datetime
import numpy as np
TAG, REPO, WORK = "p1-v1.0", "sakotiya/data266-lab2-team32", "/content/data266-lab2-team32"
from google.colab import userdata
if not os.path.isdir(WORK):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", "-b", TAG, f"https://{token}@github.com/{REPO}.git", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print({"part": 1, "tag": TAG, "commit": commit, "python": platform.python_version(),
       "gpu": gpu or "cpu", "seed": SEED, "time": datetime.datetime.now().isoformat(timespec="minutes")})

In [ ]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")   # only used if a figure is missing from the cache
except Exception:
    pass
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf==2.1.*", "chromadb==1.5.9",
                "sentence-transformers==5.7.0", "rank-bm25==0.2.2", "google-genai==2.12.1", "anthropic==2.8.0"], check=True)
from importlib.metadata import version
print({p: version(p) for p in ["pymupdf", "chromadb", "sentence-transformers", "rank-bm25", "anthropic"]})

## Data: the 3 PDFs (not in git)

In [ ]:
import shutil, glob
from part1_rag.src import common
need = [d["file"] for d in common.DOCS.values()]
drive_dir = "/content/drive/MyDrive/genai_lab2/p1_data"
if not all((common.DATA / f).exists() for f in need):
    from google.colab import drive
    drive.mount("/content/drive")
    for f in need:
        if os.path.exists(f"{drive_dir}/{f}"):
            shutil.copy(f"{drive_dir}/{f}", common.DATA / f)
missing = [f for f in need if not (common.DATA / f).exists()]
if missing:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (common.DATA / name.replace(" (1)", "")).write_bytes(data)
missing = [f for f in need if not (common.DATA / f).exists()]
assert not missing, f"missing: {missing}"
print("PDFs ready:", need)

## Ingest and index (1.1)
Pages become text chunks, figures and tables are cropped; transcriptions come from the committed cache (no API calls).

In [ ]:
from part1_rag.src import ingest, index
chunks, figures = ingest.ingest_all()
index.build_index()
print(len(chunks), "page chunks,", len(figures), "figures/tables")

In [ ]:
import json, time
from IPython.display import display, Image as IPImage, Markdown
from part1_rag.src.search import Searcher
from part1_rag.src.answer import Conversation
searcher = Searcher()
FIG_PATH = {(f["doc_id"], f["figure_id"]): f["image_path"] for f in figures}
chat = None

def new_chat():
    """Start a new conversation with empty memory (like a new conversation_id)."""
    global chat
    chat = Conversation(searcher, mode="hybrid")
    print("New conversation: empty memory.")

def ask(question, show_figures=True):
    """Answer one turn of the current conversation and show the evidence it used."""
    if chat is None:
        new_chat()
    t = time.perf_counter()
    a = chat.answer(question)
    secs = time.perf_counter() - t
    display(Markdown(f"**Q (turn {len(chat.turns)}):** {question}\n\n**A:** {a['answer']}\n\n"
                     f"**status:** `{a['status']}` · **citations:** `{', '.join(a['citations']) or 'NONE'}` · "
                     f"**figures:** `{', '.join(a['figure_references']) or 'NONE'}` · {secs:.1f} s"))
    print("retrieved evidence:", [it["id"] for it in chat.last_items][:10])
    if show_figures:
        for c in a["citations"]:
            doc, _, label = c.partition(":")
            if (doc, label) in FIG_PATH:
                print(c); display(IPImage(filename=FIG_PATH[(doc, label)], width=420))
    return a

new_chat()

## Ask questions (graders choose)
Type the grader's question in `ask("...")`. Each `ask` is the next turn of the same conversation; call `new_chat()`
before an unrelated question. Statuses: `answered`, `not_found`, `conflict`, `false_premise`, `session_override`.

In [ ]:
new_chat()
ask("According to Table B-1, what is the Okun's law slope for the Euro area, and what growth rate keeps Euro area unemployment constant?")

### Multi-turn conversation (user value kept in this conversation only)

In [ ]:
new_chat()
ask("For this chat only, assume the U.S. unemployment rate in 2013 was 6.5%. How far below the Euro area rate would that be?")
ask("Under my assumption, how much did U.S. unemployment change from 2012 to 2013?")
ask("What does the supplement itself report for that year?")

### Safety: the untrusted Appendix C (prompt injection in a chart)

In [ ]:
new_chat()
ask("What was U.S. inflation in 2009?")

### Grader's questions

In [ ]:
new_chat()
ask("TYPE THE GRADER'S QUESTION HERE")

In [ ]:
ask("TYPE THE FOLLOW-UP HERE")